# 19 — Cloud Security Architecture & Controls

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Cloud Security  
**Certifications:** AWS Certified Security – Specialty, CISM, CISA

---

## Objectives
- Design a secure cloud architecture using defence-in-depth
- Implement Zero Trust principles in a cloud environment
- Apply encryption at rest and in transit
- Harden cloud identity and access management (IAM)
- Secure containers and serverless workloads
- Perform cloud security posture management (CSPM) checks
- Map cloud controls to NIST CSF and CIS Benchmarks

## 1. Setup & Imports

In [ ]:
import json
from datetime import datetime, timezone
from collections import Counter

## 2. Defence-in-Depth for Cloud

```
  ┌─────────────────────────────────────────────────┐
  │  LAYER 7 — Data Security                        │  Encryption, DLP, Macie
  ├─────────────────────────────────────────────────┤
  │  LAYER 6 — Application Security                 │  WAF, SAST/DAST, Secrets Mgmt
  ├─────────────────────────────────────────────────┤
  │  LAYER 5 — Compute Security                     │  Patch, EDR, Container scanning
  ├─────────────────────────────────────────────────┤
  │  LAYER 4 — Identity & Access Management         │  IAM, MFA, PIM, Just-in-time
  ├─────────────────────────────────────────────────┤
  │  LAYER 3 — Network Security                     │  VPC, SGs, NACLs, Firewall, VPN
  ├─────────────────────────────────────────────────┤
  │  LAYER 2 — Monitoring & Detection               │  CloudTrail, GuardDuty, SIEM
  ├─────────────────────────────────────────────────┤
  │  LAYER 1 — Governance & Compliance              │  Policy, Standards, Risk Mgmt
  └─────────────────────────────────────────────────┘
```

### Zero Trust Principles for Cloud

| Principle | Cloud Implementation |
|-----------|---------------------|
| **Verify explicitly** | MFA + conditional access on every login |
| **Least privilege** | IAM roles with minimum required permissions; time-bound access |
| **Assume breach** | Network segmentation; lateral movement detection; micro-segmentation |
| **Never trust, always verify** | mTLS between services; no implicit trust between VMs |
| **Inspect and log everything** | All traffic logged; API calls audited via CloudTrail/Activity Log |

## 3. IAM Hardening Controls

In [ ]:
IAM_CONTROLS = [
    {'control': 'Enforce MFA on all human IAM users',                'implemented': True,  'tier': 'CRITICAL'},
    {'control': 'Delete / disable root account access keys',          'implemented': True,  'tier': 'CRITICAL'},
    {'control': 'Enable MFA on root account',                         'implemented': False, 'tier': 'CRITICAL'},
    {'control': 'Use IAM roles for EC2/Lambda — no hardcoded keys',   'implemented': True,  'tier': 'CRITICAL'},
    {'control': 'Rotate access keys every 90 days',                   'implemented': False, 'tier': 'HIGH'},
    {'control': 'Quarterly access reviews — remove unused accounts',  'implemented': False, 'tier': 'HIGH'},
    {'control': 'Enable AWS Organizations SCPs to restrict regions',  'implemented': True,  'tier': 'HIGH'},
    {'control': 'Use permission boundaries on developer roles',        'implemented': False, 'tier': 'HIGH'},
    {'control': 'Enable IAM Access Analyser',                         'implemented': True,  'tier': 'MEDIUM'},
    {'control': 'Tag all IAM resources for ownership tracking',        'implemented': False, 'tier': 'MEDIUM'},
]

passed = [c for c in IAM_CONTROLS if c['implemented']]
failed = [c for c in IAM_CONTROLS if not c['implemented']]

print('=== IAM HARDENING CHECKLIST ===\n')
for c in IAM_CONTROLS:
    icon = '✅' if c['implemented'] else '❌'
    print(f"  {icon} [{c['tier']:8}] {c['control']}")

print(f'\nScore: {len(passed)}/{len(IAM_CONTROLS)} controls implemented')
print(f'Critical gaps: {sum(1 for c in failed if c["tier"]=="CRITICAL")}')

## 4. Encryption Strategy

In [ ]:
ENCRYPTION_STRATEGY = {
    'at_rest': [
        {'resource': 'S3 Buckets',          'method': 'SSE-KMS (AES-256)',      'status': 'ENABLED',  'key_rotation': True},
        {'resource': 'RDS Databases',       'method': 'AES-256 via KMS CMK',    'status': 'ENABLED',  'key_rotation': True},
        {'resource': 'EBS Volumes',         'method': 'AES-256 via KMS',        'status': 'PARTIAL',  'key_rotation': False},
        {'resource': 'DynamoDB Tables',     'method': 'AWS Owned Key',          'status': 'ENABLED',  'key_rotation': True},
        {'resource': 'Secrets Manager',     'method': 'KMS CMK',                'status': 'ENABLED',  'key_rotation': True},
        {'resource': 'CloudWatch Logs',     'method': 'KMS CMK',                'status': 'DISABLED', 'key_rotation': False},
    ],
    'in_transit': [
        {'connection': 'HTTPS (TLS 1.2+)',  'enforced': True,  'note': 'All public endpoints'},
        {'connection': 'TLS 1.0/1.1',       'enforced': False, 'note': 'DEPRECATED — disable immediately'},
        {'connection': 'VPC to On-Prem VPN','enforced': True,  'note': 'IPSec IKEv2 AES-256'},
        {'connection': 'Internal service mTLS','enforced': False, 'note': 'Service mesh (Istio) planned'},
        {'connection': 'S3 Bucket Policy enforce HTTPS', 'enforced': True, 'note': 'aws:SecureTransport condition active'},
    ]
}

print('=== ENCRYPTION AT REST ===\n')
for r in ENCRYPTION_STRATEGY['at_rest']:
    icon = '✅' if r['status'] == 'ENABLED' else '⚠️ ' if r['status'] == 'PARTIAL' else '❌'
    rot  = '🔄' if r['key_rotation'] else '  '
    print(f"  {icon} {r['resource']:<22} {r['method']:<25} {rot} Key rotation")

print('\n=== ENCRYPTION IN TRANSIT ===\n')
for c in ENCRYPTION_STRATEGY['in_transit']:
    icon = '✅' if c['enforced'] else '❌'
    print(f"  {icon} {c['connection']:<35} {c['note']}")

## 5. Network Security — VPC Architecture

In [ ]:
# Secure 3-tier VPC architecture
VPC_ARCHITECTURE = {
    'vpc_cidr'  : '10.0.0.0/16',
    'region'    : 'us-east-1',
    'subnets'   : [
        {'name': 'public-subnet-1a',   'cidr': '10.0.1.0/24',  'az': 'us-east-1a', 'tier': 'PUBLIC',  'internet_gateway': True},
        {'name': 'public-subnet-1b',   'cidr': '10.0.2.0/24',  'az': 'us-east-1b', 'tier': 'PUBLIC',  'internet_gateway': True},
        {'name': 'private-app-1a',     'cidr': '10.0.10.0/24', 'az': 'us-east-1a', 'tier': 'PRIVATE', 'internet_gateway': False},
        {'name': 'private-app-1b',     'cidr': '10.0.11.0/24', 'az': 'us-east-1b', 'tier': 'PRIVATE', 'internet_gateway': False},
        {'name': 'private-data-1a',    'cidr': '10.0.20.0/24', 'az': 'us-east-1a', 'tier': 'DATA',    'internet_gateway': False},
        {'name': 'private-data-1b',    'cidr': '10.0.21.0/24', 'az': 'us-east-1b', 'tier': 'DATA',    'internet_gateway': False},
    ],
    'security_groups': [
        {'name': 'sg-alb',        'inbound': 'HTTPS 443 from 0.0.0.0/0',          'outbound': 'HTTP 80 to sg-app'},
        {'name': 'sg-app',        'inbound': 'HTTP 80 from sg-alb',               'outbound': 'MySQL 3306 to sg-db'},
        {'name': 'sg-db',         'inbound': 'MySQL 3306 from sg-app only',       'outbound': 'None'},
        {'name': 'sg-bastion',    'inbound': 'SSH 22 from corporate IP only',     'outbound': 'SSH 22 to sg-app'},
    ]
}

print('=== SECURE 3-TIER VPC ARCHITECTURE ===\n')
print(f"VPC CIDR: {VPC_ARCHITECTURE['vpc_cidr']}  |  Region: {VPC_ARCHITECTURE['region']}\n")

print('Subnets:')
for tier in ['PUBLIC', 'PRIVATE', 'DATA']:
    print(f'  [{tier} TIER]')
    for s in [sub for sub in VPC_ARCHITECTURE['subnets'] if sub['tier'] == tier]:
        igw = '🌐 Internet access' if s['internet_gateway'] else '🔒 No direct internet'
        print(f"    {s['name']:<24} {s['cidr']:<16} {s['az']}  {igw}")

print('\nSecurity Groups (Layered Access):')
for sg in VPC_ARCHITECTURE['security_groups']:
    print(f"  [{sg['name']:<12}]  IN: {sg['inbound'][:50]}")

## 6. Container & Serverless Security

In [ ]:
CONTAINER_SECURITY_CHECKS = [
    {'check': 'Use minimal base images (Alpine, Distroless)',         'status': True},
    {'check': 'Scan images for CVEs before deployment (Trivy/Snyk)', 'status': True},
    {'check': 'Run containers as non-root user',                      'status': False},
    {'check': 'Use read-only root filesystem',                        'status': False},
    {'check': 'Set CPU/memory limits on all containers',              'status': True},
    {'check': 'Use Kubernetes Network Policies for pod isolation',    'status': False},
    {'check': 'Enable Kubernetes RBAC with least privilege',          'status': True},
    {'check': 'Secrets via Vault/Secrets Manager — not env vars',     'status': False},
    {'check': 'Sign and verify container images (Cosign/Notary)',     'status': False},
    {'check': 'Enable runtime threat detection (Falco/Defender)',     'status': True},
]

SERVERLESS_SECURITY_CHECKS = [
    {'check': 'Lambda execution role uses least-privilege IAM',       'status': True},
    {'check': 'Function timeout and memory limits configured',        'status': True},
    {'check': 'No hardcoded secrets in function code',                'status': False},
    {'check': 'Lambda in VPC for private resource access',            'status': True},
    {'check': 'Input validation on all function event triggers',      'status': False},
    {'check': 'Dead letter queues configured for error handling',     'status': False},
]

print('=== CONTAINER SECURITY CHECKLIST ===\n')
for c in CONTAINER_SECURITY_CHECKS:
    icon = '✅' if c['status'] else '❌'
    print(f'  {icon} {c["check"]}')

cont_score = sum(1 for c in CONTAINER_SECURITY_CHECKS if c['status'])
print(f'\nContainer score: {cont_score}/{len(CONTAINER_SECURITY_CHECKS)}')

print('\n=== SERVERLESS SECURITY CHECKLIST ===\n')
for c in SERVERLESS_SECURITY_CHECKS:
    icon = '✅' if c['status'] else '❌'
    print(f'  {icon} {c["check"]}')

srv_score = sum(1 for c in SERVERLESS_SECURITY_CHECKS if c['status'])
print(f'\nServerless score: {srv_score}/{len(SERVERLESS_SECURITY_CHECKS)}')

## 7. CSPM — Cloud Security Posture Controls Mapping

In [ ]:
# Map cloud controls to NIST CSF and CIS AWS Benchmark
CONTROL_MAPPING = [
    {'control': 'Enable CloudTrail all regions',    'nist_csf': 'DE.CM-03', 'cis': '2.1',  'status': 'PASS'},
    {'control': 'CloudTrail log validation',         'nist_csf': 'PR.DS-06', 'cis': '2.2',  'status': 'FAIL'},
    {'control': 'S3 Block Public Access (account)',  'nist_csf': 'PR.AC-03', 'cis': '2.1.5','status': 'PASS'},
    {'control': 'Root MFA enabled',                  'nist_csf': 'PR.AC-01', 'cis': '1.5',  'status': 'FAIL'},
    {'control': 'No root access keys',               'nist_csf': 'PR.AC-01', 'cis': '1.4',  'status': 'PASS'},
    {'control': 'GuardDuty enabled',                 'nist_csf': 'DE.CM-01', 'cis': '3.1',  'status': 'PASS'},
    {'control': 'Security Hub enabled',              'nist_csf': 'ID.RA-01', 'cis': '3.6',  'status': 'FAIL'},
    {'control': 'VPC Flow Logs enabled',             'nist_csf': 'DE.CM-01', 'cis': '2.9',  'status': 'FAIL'},
    {'control': 'Default SG restricts all traffic',  'nist_csf': 'PR.AC-05', 'cis': '4.3',  'status': 'PASS'},
    {'control': 'RDS not publicly accessible',       'nist_csf': 'PR.AC-03', 'cis': '2.3.2','status': 'PASS'},
]

passed = [c for c in CONTROL_MAPPING if c['status'] == 'PASS']
failed = [c for c in CONTROL_MAPPING if c['status'] == 'FAIL']

print('=== CLOUD SECURITY POSTURE MANAGEMENT (CSPM) ===\n')
print(f'{"Control":<38} {"NIST CSF":<12} {"CIS":<8} Status')
print('-' * 75)
for c in CONTROL_MAPPING:
    icon = '✅' if c['status'] == 'PASS' else '❌'
    print(f"  {c['control']:<36} {c['nist_csf']:<12} {c['cis']:<8} {icon} {c['status']}")

print(f'\nPASS: {len(passed)}/{len(CONTROL_MAPPING)} | FAIL: {len(failed)}/{len(CONTROL_MAPPING)}')

report = {
    'report_generated'  : datetime.now(timezone.utc).isoformat(),
    'assessed_by'       : 'Bency (Benjamin Adjei)',
    'iam_score'         : f'{len([c for c in IAM_CONTROLS if c["implemented"]])}/{len(IAM_CONTROLS)}',
    'cspm_pass_rate'    : f'{len(passed)}/{len(CONTROL_MAPPING)}',
    'container_score'   : f'{cont_score}/{len(CONTAINER_SECURITY_CHECKS)}',
    'serverless_score'  : f'{srv_score}/{len(SERVERLESS_SECURITY_CHECKS)}',
    'top_priorities'    : [
        'Enable MFA on root account immediately (CIS 1.5)',
        'Enable VPC Flow Logs for network visibility',
        'Enable Security Hub for centralised findings',
        'Enable CloudTrail log file validation',
        'Run containers as non-root; use read-only filesystems',
        'Migrate Lambda secrets from env vars to Secrets Manager'
    ]
}
print('\n' + json.dumps(report, indent=2))

## 8. Key Takeaways

| Concept | Takeaway |
|---------|----------|
| **Defence-in-depth** | Layer controls at identity, network, compute, data — one layer failing should not mean full compromise |
| **Zero Trust** | Network location is not a trust signal in cloud — verify every request |
| **IAM is everything** | Most cloud breaches trace to IAM misconfiguration — treat it like your crown jewels |
| **Encrypt by default** | Enable encryption on every service at creation — retrofitting is harder |
| **VPC architecture** | Public subnets for load balancers only; app and data tiers stay private |
| **Container security** | Non-root, read-only filesystem, scanned images — three lowest-effort, highest-impact controls |

---
*Next: 20 — Cloud Management: Governance, Cost, Monitoring & DevSecOps*